In [1]:
import pandas as pd
import numpy as np

clean = pd.read_csv("../data/processed/occurrences_clean.csv")

clean["genus"] = (
    clean["accepted_name"]
    .astype(str)
    .str.split()
    .str[0]
)

clean.head()

,accepted_name,early_interval,late_interval,collection_no,environment,formation,lat,lng,max_ma,min_ma,genus
0,Marginalosia planata,Changhsingian,NaN,7204,NaN,Stephens,-41.299999,173.266663,254.14,251.902,Marginalosia
1,Terrakea,Changhsingian,NaN,7204,NaN,Stephens,-41.299999,173.266663,254.14,251.902,Terrakea
2,Spiriferella,Changhsingian,NaN,7204,NaN,Stephens,-41.299999,173.266663,254.14,251.902,Spiriferella
3,Martinia,Changhsingian,NaN,7204,NaN,Stephens,-41.299999,173.266663,254.14,251.902,Martinia
4,Tomiopsis,Changhsingian,NaN,7204,NaN,Stephens,-41.299999,173.266663,254.14,251.902,Tomiopsis


In [2]:
interval_order = [
    "Changhsingian",
    "Griesbachian",
    "Dienerian",
    "Smithian",
    "Spathian",
    "Aegean",
    "Bithynian",
    "Pelsonian",
    "Illyrian",
    "Anisian"
]

In [3]:
presence = pd.crosstab(
    clean["genus"],
    clean["early_interval"]
)

presence = presence.reindex(
    columns=interval_order,
    fill_value=0
)

presence = (presence > 0).astype(int)

print(presence.shape)
presence.head()

(1644, 10)


early_interval,Changhsingian,Griesbachian,Dienerian,Smithian,Spathian,Aegean,Bithynian,Pelsonian,Illyrian,Anisian
genus,,,,,,,,,,
Abichites,1,0,0,0,0,0,0,0,0,0
Abrekia,0,1,0,1,0,1,0,0,0,0
Abrekites,0,0,0,1,0,0,0,0,0,0
Abrekopsis,0,1,1,1,1,0,0,0,0,0
Acanthopecten,1,0,0,0,0,0,0,0,0,0


In [4]:
survival_rows = []

for genus in presence.index:

    for i in range(len(interval_order) - 1):

        current = interval_order[i]
        nxt = interval_order[i + 1]

        if presence.loc[genus, current] == 1:

            survival_rows.append({
                "genus": genus,
                "interval": current,
                "survived": int(
                    presence.loc[genus, nxt] == 1
                )
            })

survival_df = pd.DataFrame(survival_rows)

print(survival_df.shape)
survival_df.head()

(2525, 3)


,genus,interval,survived
0,Abichites,Changhsingian,0
1,Abrekia,Griesbachian,0
2,Abrekia,Smithian,0
3,Abrekia,Aegean,0
4,Abrekites,Smithian,0


In [5]:
survival_df["survived"].value_counts()

0    1536
1     989
Name: survived, dtype: int64

In [6]:
lazarus_records = []

for genus in presence.index:

    row = presence.loc[genus].values

    for i in range(1, len(row)-1):

        if (
            row[i-1] == 1 and
            row[i] == 0 and
            row[i+1] == 1
        ):
            lazarus_records.append({
                "genus": genus,
                "missing_interval": interval_order[i]
            })

lazarus_df = pd.DataFrame(lazarus_records)

print(lazarus_df.shape)

(196, 2)


In [7]:
summary = {
    "Genera": len(presence),
    "Intervals": len(interval_order),
    "Survival_Observations": len(survival_df),
    "Survived": int((survival_df["survived"] == 1).sum()),
    "Extinct": int((survival_df["survived"] == 0).sum()),
    "Lazarus_Events": len(lazarus_df),
    "Unique_Lazarus_Genera": lazarus_df["genus"].nunique()
}

summary

{'Genera': 1644,
 'Intervals': 10,
 'Survival_Observations': 2525,
 'Survived': 989,
 'Extinct': 1536,
 'Lazarus_Events': 196,
 'Unique_Lazarus_Genera': 161}

In [8]:
presence.to_csv(
    "../data/processed/presence_matrix.csv"
)

survival_df.to_csv(
    "../data/processed/survival_labels_genus.csv",
    index=False
)

lazarus_df.to_csv(
    "../data/processed/lazarus_taxa.csv",
    index=False
)

In [10]:
import pandas as pd

lazarus_df = pd.read_csv("../data/processed/lazarus_taxa.csv")
print(lazarus_df.columns.tolist())
print(lazarus_df.head(10))

['genus', 'missing_interval']
             genus missing_interval
0          Abrekia        Dienerian
1          Abrekia         Spathian
2  Acrochordiceras         Spathian
3         Adygella        Pelsonian
4       Allocosmia         Illyrian
5       Ammonoidea        Dienerian
6      Anagymnites        Pelsonian
7       Anomphalus        Dienerian
8  Arctohungarites         Illyrian
9          Asoella        Dienerian


In [12]:
import inspect
# This won't work directly since notebooks don't store cell source by variable,
# so instead: scroll up in that notebook and look for the cell where "lazarus"
# first appears as a column being created or merged into ml_df